# Week 7 — RFM Analysis: Role A (Data Loading & Cleaning)
Load UrbanStyle sales and customer data, check quality, merge, and clean it for RFM analysis.

In [1]:
import pandas as pd

df_sales = pd.read_csv('../data/sales.csv')
print(df_sales.shape)
df_sales.head()

(15234, 11)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [2]:
df_sales.dtypes

sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object

In [3]:
df_customers = pd.read_csv('../data/customers.csv')
print(df_customers.shape)
df_customers.head()

(3150, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2001,Eha,Aas,eha.aas@telia.ee,+372 8713 1455,Tallinn,2024-02-27,NaN,1973
1,2002,Aivar,Kõiv,aivar.koiv@outlook.com,+372 8943 8684,Haapsalu,2025-01-09,bronze,1988
2,2003,Maris,Rebane,maris.rebane@telia.ee,+372 5918 5726,Tartu,2021-02-03,NaN,1999
3,2004,Jaak,Talvik,jaak.talvik@mail.ee,+372 8554 4232,Tallinn,2023-11-12,silver,1974
4,2005,Raivo,Koppel,raivo.koppel@yahoo.com,+372 5298 4365,Tallinn,2023-05-22,bronze,2004


In [4]:
df = pd.merge(
    df_sales,
    df_customers[['customer_id', 'first_name', 'email']],
    on='customer_id',
    how='left'
)
print(df.shape)
df.head()

(15234, 13)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,email
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,NaN
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,merle.luik@mail.ee
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,liina.saar@gmail.com
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,aili.pihl@yahoo.com
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,triin.lill@telia.ee


In [5]:
print("Rows before:", len(df))
print("Duplicate invoices:", df.duplicated(subset=['invoice_id']).sum())

df = df.drop_duplicates(subset=['invoice_id'], keep='first')

print("Rows after:", len(df))

Rows before: 15234
Duplicate invoices: 5116
Rows after: 10118


In [6]:
df.isnull().sum()

sale_id              0
invoice_id           0
sale_date            0
customer_id        988
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3462
payment_method       0
first_name         988
email             1944
dtype: int64

In [7]:
print("Rows before:", len(df))

df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])
df['customer_id'] = df['customer_id'].astype(int)

print("Rows after:", len(df))
df.head()

Rows before: 10118
Rows after: 9130


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,email
0,1,INV-202301-00001,2023-01-10,2588,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,NaN
1,2,INV-202301-00002,2023-01-16,4338,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,merle.luik@mail.ee
2,3,INV-202301-00003,2023-01-05,4673,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,liina.saar@gmail.com
3,4,INV-202301-00004,2023-01-02,4677,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,aili.pihl@yahoo.com
4,5,INV-202301-00005,2023-01-13,2390,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,triin.lill@telia.ee


In [8]:
iso_dates = pd.to_datetime(df['sale_date'], format='%Y-%m-%d', errors='coerce')
eu_dates = pd.to_datetime(df['sale_date'], format='%d/%m/%Y', errors='coerce')

df['sale_date'] = iso_dates.fillna(eu_dates)

print("Unreadable dates:", df['sale_date'].isnull().sum())
print(df['sale_date'].dtype)
print("First sale:", df['sale_date'].min())
print("Last sale:", df['sale_date'].max())

Unreadable dates: 0
datetime64[us]
First sale: 2023-01-01 00:00:00
Last sale: 2026-06-28 00:00:00


In [9]:
print("Rows before:", len(df))
print("Bad prices (0 or negative):", (df['total_price'] <= 0).sum())

df = df[df['total_price'] > 0]

print("Rows after:", len(df))

Rows before: 9130
Bad prices (0 or negative): 180
Rows after: 8950


In [10]:
print("=== CLEANING REPORT ===")
print("Raw rows loaded:      15234")
print("Duplicate invoices:  -5116")
print("No customer_id:       -988")
print("Price 0 or negative:  -180")
print("Clean rows:          ", len(df))
print("Unique customers:    ", df['customer_id'].nunique())
print("Date range:          ", df['sale_date'].min().date(), "to", df['sale_date'].max().date())

=== CLEANING REPORT ===
Raw rows loaded:      15234
Duplicate invoices:  -5116
No customer_id:       -988
Price 0 or negative:  -180
Clean rows:           8950
Unique customers:     2540
Date range:           2023-01-01 to 2026-06-28


In [11]:
df.to_csv('../data/sales_clean.csv', index=False)
print("Saved:", len(df), "rows")

Saved: 8950 rows
